In [1]:
!pip install -q ucimlrepo holidays

In [12]:
import os
import numpy as np
import pandas as pd
import requests
import holidays
from google.colab import drive

drive.mount('/content/drive')

# --- Chemins (sur Google Drive pour ne rien perdre) ---
BASE_DIR      = '/content/drive/MyDrive/projet_conso'
RAW_DIR       = f'{BASE_DIR}/data/raw'
PROCESSED_DIR = f'{BASE_DIR}/data/processed'
for d in (RAW_DIR, PROCESSED_DIR):
    os.makedirs(d, exist_ok=True)

MINUTE_PATH  = f'{RAW_DIR}/household_power_minute.parquet'
WEATHER_PATH = f'{RAW_DIR}/weather_sceaux_hourly.parquet'
FINAL_PATH   = f'{PROCESSED_DIR}/dataset_hourly.parquet'

# --- Paramètres de traitement ---
MIN_VALID_MINUTES = 45
SHORT_GAP_MAX     = 3
WEEK              = 168

# --- Localisation de la maison (Sceaux) pour la météo ---
LAT, LON = 48.7786, 2.2906

# --- Groupes de colonnes ---
MEAN_COLS    = ['global_active_power', 'global_reactive_power', 'voltage', 'global_intensity']
SUM_COLS     = ['sub_1_kitchen', 'sub_2_laundry', 'sub_3_heater_ac']
MEASURE_COLS = MEAN_COLS + SUM_COLS

FINAL_COLS = MEASURE_COLS + [
    'unmetered_wh', 'hour', 'day_of_week', 'month',
    'is_weekend', 'is_holiday', 'apparent_temperature', 'is_imputed',
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
def clean_minute_data(raw: pd.DataFrame) -> pd.DataFrame:
    """Nettoie le dataset UCI brut (1 ligne = 1 minute).

    - fusionne date + time en un index datetime trié
    - convertit les valeurs en numérique ('?' -> NaN)
    - renomme les sous-compteurs
    - supprime les timestamps dupliqués
    """
    df = raw.copy()
    df.columns = df.columns.str.strip().str.lower()

    df['datetime'] = pd.to_datetime(
        df['date'].astype(str) + ' ' + df['time'].astype(str),
        format='%d/%m/%Y %H:%M:%S',
    )
    df = df.drop(columns=['date', 'time']).set_index('datetime').sort_index()

    df = df.rename(columns={
        'sub_metering_1': 'sub_1_kitchen',
        'sub_metering_2': 'sub_2_laundry',
        'sub_metering_3': 'sub_3_heater_ac',
    })
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df[~df.index.duplicated(keep='first')]
    return df[MEASURE_COLS]


# Chargement avec cache : le téléchargement UCI ne se fait qu'une fois
if os.path.exists(MINUTE_PATH):
    df_min = pd.read_parquet(MINUTE_PATH)
    print('Données minute chargées depuis Drive')
else:
    from ucimlrepo import fetch_ucirepo
    uci = fetch_ucirepo(id=235)
    df_min = clean_minute_data(uci.data.original)
    df_min.to_parquet(MINUTE_PATH)
    print('Données minute téléchargées, nettoyées et sauvegardées')

df_min.head()

Données minute chargées depuis Drive


,global_active_power,global_reactive_power,voltage,global_intensity,sub_1_kitchen,sub_2_laundry,sub_3_heater_ac
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [14]:
full_range = pd.date_range(df_min.index.min(), df_min.index.max(), freq='min')

print(f'Lignes            : {len(df_min):,}')
print(f'Période           : {df_min.index.min()} → {df_min.index.max()}')
print(f'Minutes absentes  : {len(full_range) - len(df_min):,}')
print('\nTaux de NaN par colonne (%) :')
print((df_min.isna().mean() * 100).round(2))

Lignes            : 2,075,259
Période           : 2006-12-16 17:24:00 → 2010-11-26 21:02:00
Minutes absentes  : 0

Taux de NaN par colonne (%) :
global_active_power      1.25
global_reactive_power    1.25
voltage                  1.25
global_intensity         1.25
sub_1_kitchen            1.25
sub_2_laundry            1.25
sub_3_heater_ac          1.25
dtype: float64


In [15]:
def aggregate_hourly(df_min: pd.DataFrame) -> pd.DataFrame:
    """Passe de la minute à l'heure.

    - puissance, tension, intensité : moyenne des minutes
    - sous-compteurs (Wh/minute)    : moyenne x 60 = Wh sur l'heure
      (équivaut à la somme, mais reste juste si quelques minutes manquent)
    - une heure avec moins de MIN_VALID_MINUTES mesures devient NaN
    """
    n_valid = df_min['global_active_power'].resample('h').count()

    hourly = df_min[MEASURE_COLS].resample('h').mean()
    hourly[SUM_COLS] = hourly[SUM_COLS] * 60
    hourly.loc[n_valid < MIN_VALID_MINUTES, MEASURE_COLS] = np.nan

    # On retire les heures incomplètes au tout début et à la toute fin
    target = hourly['global_active_power']
    return hourly.loc[target.first_valid_index():target.last_valid_index()]


hourly = aggregate_hourly(df_min)

print(f'Heures            : {len(hourly):,}')
print(f'Heures manquantes : {hourly["global_active_power"].isna().sum():,} '
      f'({hourly["global_active_power"].isna().mean():.2%})')
hourly.head()

Heures            : 34,587
Heures manquantes : 436 (1.26%)


,global_active_power,global_reactive_power,voltage,global_intensity,sub_1_kitchen,sub_2_laundry,sub_3_heater_ac
datetime,,,,,,,
2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0
2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0
2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0
2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0
2006-12-16 22:00:00,2.200133,0.056167,238.760000,9.523333,0.0,8.0,266.0


In [16]:
def gap_lengths(mask: pd.Series) -> pd.Series:
    """Pour chaque heure manquante, renvoie la longueur du trou auquel elle appartient (0 sinon)."""
    groups = (mask != mask.shift()).cumsum()
    return mask.groupby(groups).transform('sum').where(mask, 0)


def impute_hourly(hourly: pd.DataFrame) -> pd.DataFrame:
    """Rebouche les heures manquantes et ajoute la colonne is_imputed.

    - trous <= SHORT_GAP_MAX heures : interpolation linéaire dans le temps
    - trous plus longs              : valeur de la même heure la semaine précédente
    """
    df = hourly.copy()
    missing = df[MEASURE_COLS].isna().any(axis=1)
    gaps = gap_lengths(missing)

    # 1) Petits trous -> interpolation
    short = missing & (gaps <= SHORT_GAP_MAX)
    interpolated = df[MEASURE_COLS].interpolate(method='time')
    df.loc[short, MEASURE_COLS] = interpolated.loc[short]

    # 2) Grands trous -> semaine précédente (répété si le trou dépasse une semaine)
    for _ in range(10):
        if not df[MEASURE_COLS].isna().any().any():
            break
        df[MEASURE_COLS] = df[MEASURE_COLS].fillna(df[MEASURE_COLS].shift(WEEK))

    # 3) Sécurité : trou en tout début de série -> semaine suivante
    df[MEASURE_COLS] = df[MEASURE_COLS].fillna(df[MEASURE_COLS].shift(-WEEK))

    df['is_imputed'] = missing.astype(int)
    return df


# Rapport sur les trous avant imputation
missing = hourly[MEASURE_COLS].isna().any(axis=1)
gaps = gap_lengths(missing)
gap_sizes = gaps[missing & (gaps != gaps.shift())]
print(f'Nombre de trous        : {len(gap_sizes)}')
print(f'  dont <= {SHORT_GAP_MAX}h          : {(gap_sizes <= SHORT_GAP_MAX).sum()}')
print(f'  dont > {SHORT_GAP_MAX}h           : {(gap_sizes > SHORT_GAP_MAX).sum()}')
print(f'Plus long trou         : {gap_sizes.max()} h')

hourly = impute_hourly(hourly)
print(f'\nNaN restants           : {hourly[MEASURE_COLS].isna().sum().sum()}')
print(f'Heures imputées        : {hourly["is_imputed"].sum()} ({hourly["is_imputed"].mean():.2%})')

Nombre de trous        : 14
  dont <= 3h          : 7
  dont > 3h           : 7
Plus long trou         : 121 h

NaN restants           : 0
Heures imputées        : 436 (1.26%)


In [17]:
def add_features(hourly: pd.DataFrame) -> pd.DataFrame:
    """Ajoute unmetered_wh et les variables calendaires."""
    df = hourly.copy()

    # --- Consommation non couverte par les sous-compteurs ---
    total_wh = df['global_active_power'] * 1000
    unmetered = total_wh - df[SUM_COLS].sum(axis=1)
    n_neg = (unmetered < 0).sum()
    print(f'unmetered_wh < 0 (erreurs de mesure, mises à 0) : {n_neg}')
    df['unmetered_wh'] = unmetered.clip(lower=0)

    # --- Calendrier (calculé à partir du datetime) ---
    df['hour']        = df.index.hour
    df['day_of_week'] = df.index.dayofweek                 # 0 = lundi ... 6 = dimanche
    df['month']       = df.index.month
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

    # --- Jours fériés France ---
    years = range(df.index.year.min(), df.index.year.max() + 1)
    fr_holidays = list(holidays.France(years=years).keys())
    df['is_holiday'] = pd.Index(df.index.date).isin(fr_holidays).astype(int)

    return df


df = add_features(hourly)
print(f'Heures fériées : {df["is_holiday"].sum()}  ({df["is_holiday"].sum() // 24} jours)')
df.head()

unmetered_wh < 0 (erreurs de mesure, mises à 0) : 0
Heures fériées : 1008  (42 jours)


,global_active_power,global_reactive_power,voltage,global_intensity,sub_1_kitchen,sub_2_laundry,sub_3_heater_ac,is_imputed,unmetered_wh,hour,day_of_week,month,is_weekend,is_holiday
datetime,,,,,,,,,,,,,,
2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0,0,2217.200000,18,5,12,1,0
2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0,0,2313.233333,19,5,12,1,0
2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0,0,2261.566667,20,5,12,1,0
2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0,0,1998.466667,21,5,12,1,0
2006-12-16 22:00:00,2.200133,0.056167,238.760000,9.523333,0.0,8.0,266.0,0,1926.133333,22,5,12,1,0


In [18]:
def fetch_weather(start: str, end: str) -> pd.DataFrame:
    """Télécharge apparent_temperature horaire pour Sceaux via l'API Open-Meteo Archive."""
    response = requests.get(
        'https://archive-api.open-meteo.com/v1/archive',
        params={
            'latitude': LAT,
            'longitude': LON,
            'start_date': start,
            'end_date': end,
            'hourly': 'apparent_temperature',
            'timezone': 'GMT',
        },
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()['hourly']

    weather = pd.DataFrame({
        'datetime': pd.to_datetime(data['time']) + pd.Timedelta(hours=1),  # UTC -> UTC+1 (heure fixe)
        'apparent_temperature': data['apparent_temperature'],
    }).set_index('datetime')

    return weather[~weather.index.duplicated(keep='first')]


if os.path.exists(WEATHER_PATH):
    weather = pd.read_parquet(WEATHER_PATH)
    print('Météo chargée depuis Drive')
else:
    weather = fetch_weather(df.index.min().strftime('%Y-%m-%d'),
                            df.index.max().strftime('%Y-%m-%d'))
    weather.to_parquet(WEATHER_PATH)
    print('Météo téléchargée et sauvegardée')

print(f'{len(weather):,} heures de météo : {weather.index.min()} → {weather.index.max()}')
weather.head()

Météo chargée depuis Drive
34,608 heures de météo : 2006-12-16 01:00:00 → 2010-11-27 00:00:00


,apparent_temperature
datetime,
2006-12-16 01:00:00,1.0
2006-12-16 02:00:00,1.3
2006-12-16 03:00:00,1.5
2006-12-16 04:00:00,2.0
2006-12-16 05:00:00,2.5


In [19]:
# Contrôle de l'alignement horaire de la météo (hiver)
winter = weather[weather.index.month.isin([12, 1, 2])]
profile = winter['apparent_temperature'].groupby(winter.index.hour).mean()
print(f'Heure du minimum : {profile.idxmin()}h  (attendu ~7-8h)')
print(f'Heure du maximum : {profile.idxmax()}h  (attendu ~14-15h)')

Heure du minimum : 7h  (attendu ~7-8h)
Heure du maximum : 15h  (attendu ~14-15h)


In [20]:
df = df.join(weather, how='left')

n_missing_weather = df['apparent_temperature'].isna().sum()
print(f'Heures sans météo après fusion : {n_missing_weather}')

# Quelques heures peuvent manquer (changements d'heure) -> interpolation
df['apparent_temperature'] = df['apparent_temperature'].interpolate(
    method='time', limit_direction='both'
)

Heures sans météo après fusion : 0


In [21]:
df = df[FINAL_COLS]

# Types propres
int_cols = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'is_imputed']
df[int_cols] = df[int_cols].astype('int8')

# Vérifications
assert df.isna().sum().sum() == 0, 'Il reste des NaN !'
assert df.index.is_monotonic_increasing, 'Index non trié !'
assert not df.index.duplicated().any(), 'Timestamps dupliqués !'

df.to_parquet(FINAL_PATH)

print(f'Dataset final sauvegardé : {FINAL_PATH}')
print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Période    : {df.index.min()} → {df.index.max()}')
print(f'Imputées   : {df["is_imputed"].mean():.2%}')

# Tableau descriptif (demandé dans les bonnes pratiques)
df.describe().T.round(2)

Dataset final sauvegardé : /content/drive/MyDrive/projet_conso/data/processed/dataset_hourly.parquet
Dimensions : 34,587 lignes × 15 colonnes
Période    : 2006-12-16 18:00:00 → 2010-11-26 20:00:00
Imputées   : 1.26%


,count,mean,std,min,25%,50%,75%,max
global_active_power,34587.0,1.09,0.90,0.12,0.34,0.80,1.58,6.56
global_reactive_power,34587.0,0.12,0.07,0.01,0.08,0.11,0.15,0.77
voltage,34587.0,240.85,3.00,225.83,239.20,240.99,242.69,251.90
global_intensity,34587.0,4.62,3.75,0.50,1.51,3.41,6.59,28.38
sub_1_kitchen,34587.0,67.27,212.26,0.00,0.00,0.00,0.00,2902.00
sub_2_laundry,34587.0,77.73,251.11,0.00,0.00,20.00,39.00,2786.00
sub_3_heater_ac,34587.0,387.18,440.70,0.00,39.00,99.00,850.00,1293.00
unmetered_wh,34587.0,556.77,495.04,82.77,260.73,343.80,663.48,3890.23
hour,34587.0,11.50,6.92,0.00,6.00,12.00,18.00,23.00
day_of_week,34587.0,3.00,2.00,0.00,1.00,3.00,5.00,6.00


In [22]:
df.loc['2008-01-01'].round(2)

,global_active_power,global_reactive_power,voltage,global_intensity,sub_1_kitchen,sub_2_laundry,sub_3_heater_ac,unmetered_wh,hour,day_of_week,month,is_weekend,is_holiday,apparent_temperature,is_imputed
datetime,,,,,,,,,,,,,,,
2008-01-01 00:00:00,1.60,0.05,241.55,6.58,0.0,0.0,1074.0,522.93,0,1,1,0,1,2.5,0
2008-01-01 01:00:00,1.62,0.09,240.83,6.70,0.0,27.0,1068.0,524.57,1,1,1,0,1,2.5,0
2008-01-01 02:00:00,2.47,0.07,242.75,10.19,0.0,0.0,1084.0,1389.73,2,1,1,0,1,2.5,0
2008-01-01 03:00:00,2.73,0.08,241.91,11.22,0.0,28.0,1077.0,1626.13,3,1,1,0,1,2.1,0
2008-01-01 04:00:00,2.43,0.06,241.07,9.99,0.0,0.0,1069.0,1362.13,4,1,1,0,1,2.0,0
2008-01-01 05:00:00,2.48,0.06,240.51,10.20,0.0,2.0,1065.0,1412.67,5,1,1,0,1,1.8,0
2008-01-01 06:00:00,1.69,0.07,241.37,6.90,0.0,26.0,341.0,1323.20,6,1,1,0,1,1.5,0
2008-01-01 07:00:00,1.33,0.06,240.99,5.44,0.0,0.0,0.0,1332.13,7,1,1,0,1,1.1,0
2008-01-01 08:00:00,1.38,0.11,241.84,5.62,0.0,28.0,0.0,1347.17,8,1,1,0,1,1.6,0
